# Step 2 — Build the Historical Outage Panel

**Objective:** Create a county x day panel of power outage metrics from EAGLE-I (2018-2024).

**Data source:** EAGLE-I Power Outage Data, ORNL — download annual CSVs from OSTI.

**Output:** `data/processed/outage_panel_ercot.csv` and `outage_panel_caiso.csv`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from config.settings import PROCESSED, ERCOT_FIPS, CAISO_FIPS
from src.data.eagle_i import build_ercot_panel, build_caiso_panel


## 2.1 Build ERCOT outage panel

Requires EAGLE-I annual CSVs in `data/raw/eagle_i/`. Download from ORNL OSTI (see `scripts/download_data.py`).

In [ ]:
try:
    ercot_outage = build_ercot_panel()
    print(f'ERCOT panel shape: {ercot_outage.shape}')
    print(ercot_outage.head())
except FileNotFoundError as e:
    print(f'Data not yet available:\n{e}')
    ercot_outage = None


## 2.2 Build CAISO outage panel

In [ ]:
try:
    caiso_outage = build_caiso_panel()
    print(f'CAISO panel shape: {caiso_outage.shape}')
    print(caiso_outage.head())
except FileNotFoundError as e:
    print(f'Data not yet available:\n{e}')
    caiso_outage = None


## 2.3 Data quality check

Flag county-years with low EAGLE-I coverage (< 80 intervals/day).

In [ ]:
if ercot_outage is not None:
    low_cov = ercot_outage['low_coverage_flag'].mean()
    print(f'ERCOT: {low_cov:.1%} of county-days have low coverage')
    print('\nOutage fraction distribution:')
    print(ercot_outage['outage_fraction'].describe())


## 2.4 Summary statistics — event rates by year

In [ ]:
if ercot_outage is not None:
    event_rate = ercot_outage['outage_event_flag'].mean()
    print(f'ERCOT outage event rate: {event_rate:.3%}')
    df_reset = ercot_outage.reset_index()
    df_reset['year'] = df_reset['date'].dt.year
    yearly = df_reset.groupby('year')['total_customer_hours'].mean()
    print('\nMean customer-hours per county-day by year:')
    print(yearly)


## 2.5 Save processed panels

In [ ]:
if ercot_outage is not None:
    ercot_outage.to_csv(PROCESSED['outage_ercot'])
    print('Saved:', PROCESSED['outage_ercot'])
if caiso_outage is not None:
    caiso_outage.to_csv(PROCESSED['outage_caiso'])
    print('Saved:', PROCESSED['outage_caiso'])
